# LeetCode 503 — Next Greater Element II bằng Python

Link đề: https://leetcode.com/problems/next-greater-element-ii/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Monotonic Stack

Skeleton LeetCode:

```python
class Solution:
    def nextGreaterElements(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: phần phát biểu đề được dịch sang tiếng Việt từ đề gốc; mọi đáp án trong notebook được chạy lại và đối chiếu với một oracle vét cạn độc lập (xoay mảng rồi tìm phần tử lớn hơn đầu tiên).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với mỗi phần tử của mảng **vòng** (sau phần tử cuối lại là phần tử đầu), tìm **số lớn hơn thật sự đầu tiên** gặp được khi đi tiếp theo chiều vòng; không có thì trả `-1`.
2. Nắm công cụ Python: toán tử `%` để "quay vòng" chỉ số, `list` dùng làm **ngăn xếp** (`append`, `pop`, `[-1]`), vòng `while` để pop nhiều lần, `[-1] * n`, cắt lát để xoay mảng, `max` / `index`.
3. Đi từ **vét cạn `O(n^2)`** → **ngăn xếp đơn điệu duyệt xuôi 2 vòng (lưu chỉ số)** → **ngăn xếp đơn điệu duyệt ngược 2 vòng (lưu giá trị)** → **xoay mảng để số lớn nhất ở cuối (một vòng)** → **nhảy theo chuỗi `next` (pointer jumping)**.
4. Hiểu mấu chốt: phần tử đang **chờ** số lớn hơn xếp trong ngăn xếp theo thứ tự **không tăng** từ đáy lên đỉnh; một số mới `x` "giải phóng" mọi phần tử nhỏ hơn `x` ở đỉnh, mỗi phần tử vào/ra ngăn xếp **một lần** ⇒ `O(n)`; và tính tuần hoàn xử lý bằng cách duyệt **2n** vị trí với chỉ số `j % n`.
5. Tránh các lỗi kinh điển: dùng `<=` thay `<` (coi số bằng nhau là lớn hơn), chỉ duyệt một vòng, quên `% n` (IndexError), nhầm chiều so sánh ở bản duyệt ngược, thao tác sai với chỉ số và giá trị trong ngăn xếp.

## 2. Đề bài & ví dụ

Cho mảng số nguyên **vòng** `nums` (phần tử kế tiếp của `nums[n-1]` là `nums[0]`). Với mỗi phần tử, hãy trả về **số lớn hơn kế tiếp** (next greater number).

**Số lớn hơn kế tiếp** của `x` là số **lớn hơn `x` đầu tiên** gặp được khi duyệt theo thứ tự kế tiếp trong mảng; được phép tìm vòng quanh. Nếu không tồn tại thì trả `-1` cho số đó.

**Ví dụ 1:** `nums = [1,2,1]` → `[2,-1,2]`.

- `1` đầu tiên: số lớn hơn kế tiếp là `2`.
- `2`: không có số nào lớn hơn nó (kể cả khi vòng lại) ⇒ `-1`.
- `1` thứ hai: tìm vòng — đi tới `nums[0] = 1` (không lớn hơn), rồi `nums[1] = 2` ⇒ `2`.

**Ví dụ 2:** `nums = [1,2,3,4,3]` → `[2,3,4,-1,4]`.

**Ràng buộc:**

- `1 <= nums.length <= 10^4`
- `-10^9 <= nums[i] <= 10^9`

In [1]:
import random
import time
from typing import List

examples = [
    ([1, 2, 1], [2, -1, 2]),
    ([1, 2, 3, 4, 3], [2, 3, 4, -1, 4]),
]
for nums, expected in examples:
    print(nums, "->", expected)

[1, 2, 1] -> [2, -1, 2]
[1, 2, 3, 4, 3] -> [2, 3, 4, -1, 4]


### 2.1. Đọc kỹ đề

- **"Lớn hơn" là lớn hơn thật sự (`>`)**, không phải `>=`. Số bằng nhau **không** tính. Ví dụ `[1,1,1]` → `[-1,-1,-1]`.
- "Đầu tiên theo thứ tự duyệt": ta xét `nums[i+1], nums[i+2], ..., nums[n-1], nums[0], nums[1], ..., nums[i-1]` rồi dừng (không xét lại chính `nums[i]`). Một số không thể là "số lớn hơn" của chính nó.
- Với mảng vòng, **phần tử lớn nhất** luôn có kết quả `-1` (không có gì lớn hơn). Mọi phần tử **khác phần tử lớn nhất** luôn có đáp án (vì phần tử lớn nhất sẽ gặp khi đi vòng đủ).
- `n <= 10^4` ⇒ `O(n^2) = 10^8` phép: Python chậm với mức đó; mong muốn `O(n)` hoặc `O(n log n)`.
- Giá trị tới `±10^9` (có cả số âm) nên không thể dùng giá trị làm chỉ số của mảng đếm; và vì đáp án hợp lệ có thể âm, ta phân biệt "không có" bằng `-1` chỉ vì đề quy định như vậy (cẩn thận khi tự thêm ví dụ có số `-1`: nó vẫn là giá trị bình thường).

## 3. Kiến thức nền

### 3.1. Chỉ số vòng bằng `%`

Mảng vòng có `n` phần tử thì sau chỉ số `n - 1` là `0`. Công thức: **chỉ số vòng** = `(i + bước) % n`. `%` cho phần dư; `(n - 1 + 1) % n = 0`, `(n + 2) % n = 2`.

In [2]:
nums = [10, 20, 30, 40]
n = len(nums)
i = 2
for step in range(n):
    j = (i + step) % n
    print(f"bước {step}: j = ({i} + {step}) % {n} = {j} -> {nums[j]}")

bước 0: j = (2 + 0) % 4 = 2 -> 30
bước 1: j = (2 + 1) % 4 = 3 -> 40
bước 2: j = (2 + 2) % 4 = 0 -> 10
bước 3: j = (2 + 3) % 4 = 1 -> 20


### 3.2. Ngăn xếp (stack) bằng `list`

Ngăn xếp: **vào sau, ra trước** (LIFO). Với `list` của Python:

- `stk.append(x)`: đẩy `x` lên đỉnh (`O(1)`);
- `stk.pop()`: lấy và trả phần tử ở đỉnh (`O(1)`);
- `stk[-1]`: xem đỉnh (không lấy);
- `while stk:` hoặc `if stk:` kiểm tra ngăn xếp không rỗng (list rỗng là `False`).

In [3]:
stk = []
stk.append(5)
stk.append(3)
stk.append(8)
print(stk, "đỉnh =", stk[-1])
print("pop ->", stk.pop(), stk)
print("rỗng không?", not stk)
stk.clear()
print("rỗng không?", not stk)

[5, 3, 8] đỉnh = 8
pop -> 8 [5, 3]
rỗng không? False
rỗng không? True


### 3.3. Ngăn xếp đơn điệu (monotonic stack)

Là ngăn xếp mà ta **giữ thứ tự đơn điệu** (ví dụ giảm dần từ đáy lên đỉnh) bằng cách trước khi `append` phần tử mới ta `pop` hết các phần tử "phá thứ tự". Mỗi lần `pop` là một "sự kiện có ích": phần tử bị pop vừa tìm được "số lớn hơn kế tiếp" của nó chính là phần tử đang chuẩn bị `append`.

In [4]:
def demo_monotonic(arr):
    stk = []                       # giữ giá trị theo thứ tự không tăng
    for x in arr:
        popped = []
        while stk and stk[-1] < x:
            popped.append(stk.pop())
        stk.append(x)
        print(f"x={x}: pop {popped}, stack = {stk}")


demo_monotonic([5, 3, 4, 2, 6])

x=5: pop [], stack = [5]
x=3: pop [], stack = [5, 3]
x=4: pop [3], stack = [5, 4]
x=2: pop [], stack = [5, 4, 2]
x=6: pop [2, 4, 5], stack = [6]


### 3.4. `[-1] * n` và gán theo chỉ số

`[-1] * n` tạo list `n` phần tử đều `-1` (an toàn cho số, vì số là bất biến). Ta dùng làm bảng kết quả khởi tạo "chưa có số lớn hơn": phần tử nào tìm được thì ghi đè. (Chú ý: nếu phần tử là list thì `[[]] * n` cùng tham chiếu — không dùng!)

### 3.5. Xoay mảng bằng cắt lát; `max` và `index`

`nums[k:] + nums[:k]` đưa phần tử `k` lên đầu (xoay trái `k`). `max(nums)` cho giá trị lớn nhất; `nums.index(v)` cho chỉ số xuất hiện đầu tiên của `v`.

In [5]:
nums = [1, 2, 1, 5, 3]
k = nums.index(max(nums))
print("max =", max(nums), "ở chỉ số", k)
print("xoay để phần tử sau max lên đầu:", nums[k + 1:] + nums[:k + 1])
print("xoay trái 2:", nums[2:] + nums[:2])

max = 5 ở chỉ số 3
xoay để phần tử sau max lên đầu: [3, 1, 2, 1, 5]
xoay trái 2: [1, 5, 3, 1, 2]


## 4. Ý tưởng nền tảng: tính tay

`nums = [1, 2, 3, 4, 3]` (vòng). Với mỗi phần tử, đọc dãy "đi tiếp theo vòng" rồi tìm số lớn hơn đầu tiên:

| `i` | `nums[i]` | dãy đi tiếp (không gồm chính nó) | số lớn hơn đầu tiên |
|---|---|---|---|
| 0 | 1 | 2, 3, 4, 3 | **2** |
| 1 | 2 | 3, 4, 3, 1 | **3** |
| 2 | 3 | 4, 3, 1, 2 | **4** |
| 3 | 4 | 3, 1, 2, 3 | không có ⇒ **-1** |
| 4 | 3 | 1, 2, 3, 4 | **4** (vòng qua đầu mảng: `1, 2, 3` không lớn hơn 3, rồi gặp `4`) |

Kết quả `[2, 3, 4, -1, 4]` ✓.

Nhận xét: để đọc "đi tiếp theo vòng" ta chỉ cần **đọc mảng hai lần liên tiếp**: `nums + nums = [1,2,3,4,3,1,2,3,4,3]`. Từ vị trí `i`, ta đọc tối đa `n - 1` phần tử sau nó trong mảng đôi.

In [6]:
nums = [1, 2, 3, 4, 3]
n = len(nums)
doubled = nums + nums
print("mảng đôi:", doubled)
for i in range(n):
    window = doubled[i + 1: i + n]                       # n - 1 phần tử kế tiếp
    ans = next((x for x in window if x > nums[i]), -1)
    print(f"i={i} nums[i]={nums[i]} cửa sổ={window} -> {ans}")

mảng đôi: [1, 2, 3, 4, 3, 1, 2, 3, 4, 3]
i=0 nums[i]=1 cửa sổ=[2, 3, 4, 3] -> 2
i=1 nums[i]=2 cửa sổ=[3, 4, 3, 1] -> 3
i=2 nums[i]=3 cửa sổ=[4, 3, 1, 2] -> 4
i=3 nums[i]=4 cửa sổ=[3, 1, 2, 3] -> -1
i=4 nums[i]=3 cửa sổ=[1, 2, 3, 4] -> 4


## 5. Cách 1 — Brute Force: với mỗi phần tử, quét tối đa `n - 1` bước vòng

**Ý tưởng.** Với mỗi `i`, thử `step = 1, 2, ..., n - 1`: `j = (i + step) % n`; gặp `nums[j] > nums[i]` đầu tiên thì ghi nhận và dừng. Nếu quét hết mà không gặp thì `-1`.

**Code demo:**

In [7]:
def next_greater_brute(nums):
    n = len(nums)
    ans = [-1] * n
    for i in range(n):
        for step in range(1, n):
            j = (i + step) % n
            if nums[j] > nums[i]:
                ans[i] = nums[j]
                break
    return ans


for nums, expected in examples:
    got = next_greater_brute(nums)
    print(nums, "->", got)
    assert got == expected

[1, 2, 1] -> [2, -1, 2]
[1, 2, 3, 4, 3] -> [2, 3, 4, -1, 4]


**Code nộp LeetCode (Cách 1):**

In [8]:
class SolutionBrute:
    def nextGreaterElements(self, nums: List[int]) -> List[int]:
        n = len(nums)
        ans = [-1] * n
        for i in range(n):
            for step in range(1, n):
                j = (i + step) % n
                if nums[j] > nums[i]:
                    ans[i] = nums[j]
                    break
        return ans


for nums, expected in examples:
    assert SolutionBrute().nextGreaterElements(nums) == expected
print("Cách 1 OK")

Cách 1 OK


**Độ phức tạp.** Hai vòng lồng nhau: tệ nhất `n · (n - 1)` ⇒ **`O(n^2)`** thời gian, **`O(1)`** bộ nhớ phụ (ngoài kết quả). Trường hợp xấu: mảng giảm dần hoặc toàn số bằng nhau (mỗi phần tử quét hết).

**Nút thắt.** Khi quét cho `nums[i]`, ta đi qua nhiều phần tử mà **khi quét cho `nums[i+1]` ta lại đi qua** chính chúng. Thông tin bị bỏ phí. Ta cần một cấu trúc nhớ "những phần tử **đang chờ** số lớn hơn" để mỗi số mới **giải phóng** hàng loạt phần tử chờ cùng lúc.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — "Đang chờ".** Duyệt từ trái sang phải. Phần tử `nums[i]` vừa duyệt **chưa biết** số lớn hơn kế tiếp, nên cho vào danh sách **đang chờ**. Khi gặp số mới `x`:

- mọi phần tử đang chờ mà **nhỏ hơn `x`** thì `x` chính là số lớn hơn kế tiếp của chúng (vì `x` là số đầu tiên lớn hơn chúng sau khi chúng bắt đầu chờ);
- sau đó `x` cũng vào danh sách chờ.

**Bước 2 — Danh sách chờ là ngăn xếp không tăng.** Nếu hai phần tử đang chờ `a` (trước) và `b` (sau) mà `a < b`, thì khi `b` vào, `a` bị giải phóng ngay (vì `b > a`) — nên `a` không thể còn trong danh sách khi `b` vào. Vậy các phần tử chờ luôn xếp **không tăng** từ cũ nhất (đáy) đến mới nhất (đỉnh); phần tử cần giải phóng luôn nằm ở **đỉnh**. Dùng ngăn xếp: mỗi phần tử vào một lần, ra một lần ⇒ **`O(n)`**.

**Bước 3 — Tính tuần hoàn.** Với những phần tử còn chờ sau lượt đầu (không có số lớn hơn ở bên phải), cần xét tiếp từ đầu mảng. Ta duyệt **`2n` vị trí** (`j = 0..2n-1`, giá trị `nums[j % n]`), nhưng chỉ cho **`j < n`** vào ngăn xếp (vòng thứ hai chỉ để "giải phóng", không cần thêm chờ nữa).

**Cách khác để xử lý vòng:** duyệt **ngược** từ `2n-1` về `0`; hoặc **xoay** để số lớn nhất ở cuối thì không cần vòng thứ hai.

## 7. Các cách tiếp theo

### Cách 2 — Ngăn xếp đơn điệu duyệt xuôi, lưu chỉ số, 2 vòng (chốt nộp)

**Giải thích từng đoạn.**

1. `ans = [-1] * n`; `stk = []` chứa **chỉ số** các phần tử đang chờ (giá trị `nums[stk[...]]` không tăng từ đáy lên đỉnh).
2. Duyệt `j` từ `0` đến `2n - 1`; `x = nums[j % n]`.
3. `while stk and nums[stk[-1]] < x:` — đỉnh nhỏ hơn `x` ⇒ `ans[stk.pop()] = x`.
4. Nếu `j < n`: `stk.append(j)` (chỉ vòng đầu cho phần tử vào chờ).

Ta lưu **chỉ số** (không phải giá trị) vì khi pop phải biết **ghi vào vị trí nào** của `ans`.

In [9]:
def next_greater_forward(nums):
    n = len(nums)
    ans = [-1] * n
    stk = []                                   # chỉ số các phần tử đang chờ
    for j in range(2 * n):
        x = nums[j % n]
        while stk and nums[stk[-1]] < x:       # strict: bằng nhau KHÔNG tính là lớn hơn
            ans[stk.pop()] = x
        if j < n:
            stk.append(j)
    return ans


for nums, expected in examples:
    assert next_greater_forward(nums) == expected
print(next_greater_forward([1, 2, 3, 4, 3]))

[2, 3, 4, -1, 4]


In [10]:
class Solution2:
    def nextGreaterElements(self, nums: List[int]) -> List[int]:
        n = len(nums)
        ans = [-1] * n
        stk: List[int] = []
        for j in range(2 * n):
            x = nums[j % n]
            while stk and nums[stk[-1]] < x:
                ans[stk.pop()] = x
            if j < n:
                stk.append(j)
        return ans


for nums, expected in examples:
    assert Solution2().nextGreaterElements(nums) == expected
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp.** Mỗi chỉ số `0..n-1` vào ngăn xếp tối đa một lần và ra tối đa một lần; vòng `for` `2n` lần ⇒ **`O(n)`** thời gian (tổng số `pop` `<= n`). Bộ nhớ phụ **`O(n)`** cho ngăn xếp.

### Cách 3 — Ngăn xếp đơn điệu duyệt ngược, lưu giá trị, 2 vòng

**Ý tưởng khác.** Duyệt **từ phải sang trái** trên chỉ số "mảng đôi" `j = 2n-1, ..., 0`. Ngăn xếp giữ **các giá trị ứng viên** cho "số lớn hơn kế tiếp" của các phần tử đứng bên trái: luôn giữ ngăn xếp **giảm chặt từ đáy lên đỉnh**, đỉnh là ứng viên gần nhất.

1. Với `x = nums[j % n]`: `while stk and stk[-1] <= x: stk.pop()` — bỏ những ứng viên **không lớn hơn** `x` (chúng bị `x` che khuất: với mọi phần tử bên trái, `x` ở gần hơn và không nhỏ hơn).
2. Nếu `j < n` (ở nửa đầu, ta đang ghi đáp án): `ans[j] = stk[-1] if stk else -1`.
3. `stk.append(x)`.

Vòng đầu tiên (`j >= n`) chỉ dùng để **nạp** ứng viên từ "phần vòng lại" vào ngăn xếp trước khi xử lý các `j < n`.

In [11]:
def next_greater_backward(nums):
    n = len(nums)
    ans = [-1] * n
    stk = []                                   # giá trị ứng viên, giảm chặt từ đáy lên đỉnh
    for j in range(2 * n - 1, -1, -1):
        x = nums[j % n]
        while stk and stk[-1] <= x:            # ứng viên không lớn hơn x -> bỏ
            stk.pop()
        if j < n and stk:
            ans[j] = stk[-1]
        stk.append(x)
    return ans


for nums, expected in examples:
    assert next_greater_backward(nums) == expected
print(next_greater_backward([1, 2, 3, 4, 3]))

[2, 3, 4, -1, 4]


In [12]:
class Solution3:
    def nextGreaterElements(self, nums: List[int]) -> List[int]:
        n = len(nums)
        ans = [-1] * n
        stk: List[int] = []
        for j in range(2 * n - 1, -1, -1):
            x = nums[j % n]
            while stk and stk[-1] <= x:
                stk.pop()
            if j < n and stk:
                ans[j] = stk[-1]
            stk.append(x)
        return ans


for nums, expected in examples:
    assert Solution3().nextGreaterElements(nums) == expected
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp.** Mỗi vị trí `j` push một lần và pop tối đa một lần ⇒ **`O(n)`** thời gian (hằng số 2 do duyệt `2n` vị trí); bộ nhớ phụ **`O(n)`**.

### Cách 4 — Xoay để số lớn nhất ở cuối, chỉ một vòng

**Ý tưởng.** Tính tuần hoàn rắc rối chỉ vì có thể phải "vòng qua đầu". Nếu ta **xoay mảng** sao cho một phần tử **lớn nhất** nằm ở **cuối**, thì:

- mọi phần tử **nhỏ hơn max** có số lớn hơn kế tiếp **ở bên phải của nó** trong mảng đã xoay (ít nhất là phần tử max ở cuối), không cần vòng lại;
- mọi phần tử **bằng max** có đáp án `-1`.

Vậy chỉ cần chạy bài "next greater **không vòng**" một lượt trên mảng xoay, rồi ánh xạ chỉ số về mảng gốc.

**Giải thích từng đoạn.**

1. `k = nums.index(max(nums))` — một vị trí của giá trị lớn nhất.
2. `start = (k + 1) % n`. Mảng xoay: `rot[r] = nums[(start + r) % n]`; phần tử cuối `rot[n-1] = nums[k]` là max.
3. Chạy ngăn xếp một lượt trên `rot` (lưu chỉ số `r`): pop khi `rot[stk[-1]] < rot[r]`, ghi kết quả vào `ans[(start + idx) % n]`.
4. Phần tử còn lại trong ngăn xếp (các max) giữ `-1`.

In [13]:
def next_greater_rotate(nums):
    n = len(nums)
    k = nums.index(max(nums))
    start = (k + 1) % n                        # sau phép xoay, phần tử max đứng cuối
    ans = [-1] * n
    stk = []                                   # chỉ số trong mảng đã xoay
    for r in range(n):
        x = nums[(start + r) % n]
        while stk and nums[(start + stk[-1]) % n] < x:
            ans[(start + stk.pop()) % n] = x   # ánh xạ về chỉ số gốc
        stk.append(r)
    return ans


for nums, expected in examples:
    assert next_greater_rotate(nums) == expected
print(next_greater_rotate([1, 2, 1]))

[2, -1, 2]


In [14]:
class Solution4:
    def nextGreaterElements(self, nums: List[int]) -> List[int]:
        n = len(nums)
        start = (nums.index(max(nums)) + 1) % n
        ans = [-1] * n
        stk: List[int] = []
        for r in range(n):
            x = nums[(start + r) % n]
            while stk and nums[(start + stk[-1]) % n] < x:
                ans[(start + stk.pop()) % n] = x
            stk.append(r)
        return ans


for nums, expected in examples:
    assert Solution4().nextGreaterElements(nums) == expected
print("Cách 4 OK")

Cách 4 OK


**Độ phức tạp.** Một vòng `n` bước, mỗi phần tử push/pop một lần ⇒ **`O(n)`** thời gian (hằng số nhỏ hơn: `n` thay vì `2n`; thêm `O(n)` để tìm max). Bộ nhớ phụ **`O(n)`**.

### Cách 5 — Nhảy theo chuỗi `next` (pointer jumping), không dùng ngăn xếp tường minh

**Ý tưởng.** Xét mảng đôi `arr = nums + nums` (độ dài `2n`). Với mỗi vị trí `i` tính `nxt[i]` = **vị trí** của số lớn hơn kế tiếp trong `arr` (hoặc `2n` nếu không có). Duyệt `i` từ phải sang trái:

- bắt đầu từ `j = i + 1`;
- khi `j < 2n` và `arr[j] <= arr[i]`: **nhảy** `j = nxt[j]` (các phần tử giữa `j` và `nxt[j]` đều `<= arr[j] <= arr[i]` nên bỏ qua an toàn);
- `nxt[i] = j`.

Rồi `ans[i] = arr[nxt[i]]` nếu `nxt[i] < 2n`, ngược lại `-1`, cho `i < n`. (Ngăn xếp thực chất đã "ẩn" trong chuỗi con trỏ `nxt`.)

**Lưu ý về đúng đắn với vòng:** với `i < n`, "số lớn hơn kế tiếp" trong mảng đôi là số lớn hơn **đầu tiên** trong `arr[i+1 .. 2n-1]` — bao gồm cả vòng lại, và vì `arr[i + n] = arr[i]` không lớn hơn chính nó nên kết quả đúng như đề.

In [15]:
def next_greater_jump(nums):
    n = len(nums)
    arr = nums + nums
    m = 2 * n
    nxt = [m] * m
    for i in range(m - 1, -1, -1):
        j = i + 1
        while j < m and arr[j] <= arr[i]:
            j = nxt[j]                         # nhảy qua cả đoạn các phần tử không lớn hơn arr[j]
        nxt[i] = j
    return [arr[nxt[i]] if nxt[i] < m else -1 for i in range(n)]


for nums, expected in examples:
    assert next_greater_jump(nums) == expected
print(next_greater_jump([1, 2, 3, 4, 3]))

[2, 3, 4, -1, 4]


In [16]:
class Solution5:
    def nextGreaterElements(self, nums: List[int]) -> List[int]:
        n = len(nums)
        arr = nums + nums
        m = 2 * n
        nxt = [m] * m
        for i in range(m - 1, -1, -1):
            j = i + 1
            while j < m and arr[j] <= arr[i]:
                j = nxt[j]
            nxt[i] = j
        return [arr[nxt[i]] if nxt[i] < m else -1 for i in range(n)]


for nums, expected in examples:
    assert Solution5().nextGreaterElements(nums) == expected
print("Cách 5 OK")

Cách 5 OK


**Độ phức tạp.** Mỗi `nxt[i]` được tính bằng cách đi dọc chuỗi con trỏ; tổng số bước nhảy **amortized `O(n)`** (cùng lý do với ngăn xếp đơn điệu: mỗi vị trí bị "nhảy qua" bởi một vị trí bên trái rồi không bao giờ bị thăm lại từ bên trái nữa — chính là các phần tử bị pop). Bộ nhớ **`O(n)`** (mảng đôi + `nxt`, `4n` ô).

## 8. TRACE từng bước

**Trace Cách 2** với `nums = [1, 2, 1]` (`n = 3`, duyệt `j = 0..5`):

| `j` | `j % 3` | `x` | pop (gán `ans`) | push? | `stk` (chỉ số) | `ans` |
|---|---|---|---|---|---|---|
| 0 | 0 | 1 | — | có (0) | `[0]` | `[-1,-1,-1]` |
| 1 | 1 | 2 | `nums[0]=1<2` ⇒ `ans[0]=2` | có (1) | `[1]` | `[2,-1,-1]` |
| 2 | 2 | 1 | `nums[1]=2<1`? không | có (2) | `[1,2]` | `[2,-1,-1]` |
| 3 | 0 | 1 | `nums[2]=1<1`? không (bằng nhau!) | không | `[1,2]` | `[2,-1,-1]` |
| 4 | 1 | 2 | `nums[2]=1<2` ⇒ `ans[2]=2`; `nums[1]=2<2`? không | không | `[1]` | `[2,-1,2]` |
| 5 | 2 | 1 | — | không | `[1]` | `[2,-1,2]` |

Kết quả `[2, -1, 2]` ✓. Chỉ số `1` (giá trị `2`, là max) còn trong ngăn xếp ⇒ giữ `-1`.

Hàm `trace_forward` in bảng này cho bất kỳ mảng nào; `trace_rotate` in phép xoay của Cách 4.

In [17]:
def trace_forward(nums):
    n = len(nums)
    ans = [-1] * n
    stk = []
    print("nums =", nums)
    print(f"{'j':>2} | {'j%n':>3} | {'x':>3} | {'pop (chỉ số -> gán)':<26} | {'push':<6} | {'stk':<16} | ans")
    for j in range(2 * n):
        x = nums[j % n]
        pops = []
        while stk and nums[stk[-1]] < x:
            idx = stk.pop()
            ans[idx] = x
            pops.append(f"{idx}->{x}")
        pushed = ""
        if j < n:
            stk.append(j)
            pushed = f"+{j}"
        print(f"{j:>2} | {j % n:>3} | {x:>3} | {', '.join(pops) or '-':<26} | {pushed:<6} | {str(stk):<16} | {ans}")
    print("kết quả:", ans)
    return ans


r = trace_forward([1, 2, 1])
print()
r = trace_forward([5, 3, 4, 1, 2])

nums = [1, 2, 1]
 j | j%n |   x | pop (chỉ số -> gán)        | push   | stk              | ans
 0 |   0 |   1 | -                          | +0     | [0]              | [-1, -1, -1]
 1 |   1 |   2 | 0->2                       | +1     | [1]              | [2, -1, -1]
 2 |   2 |   1 | -                          | +2     | [1, 2]           | [2, -1, -1]
 3 |   0 |   1 | -                          |        | [1, 2]           | [2, -1, -1]
 4 |   1 |   2 | 2->2                       |        | [1]              | [2, -1, 2]
 5 |   2 |   1 | -                          |        | [1]              | [2, -1, 2]
kết quả: [2, -1, 2]

nums = [5, 3, 4, 1, 2]
 j | j%n |   x | pop (chỉ số -> gán)        | push   | stk              | ans
 0 |   0 |   5 | -                          | +0     | [0]              | [-1, -1, -1, -1, -1]
 1 |   1 |   3 | -                          | +1     | [0, 1]           | [-1, -1, -1, -1, -1]
 2 |   2 |   4 | 1->4                       | +2     | [0, 2]           | [-1,

In [18]:
def trace_rotate(nums):
    n = len(nums)
    k = nums.index(max(nums))
    start = (k + 1) % n
    rot = [nums[(start + r) % n] for r in range(n)]
    print("nums =", nums, " max =", nums[k], "ở chỉ số", k)
    print("start =", start, " mảng xoay rot =", rot, "(max đứng cuối)")
    ans = [-1] * n
    stk = []
    for r in range(n):
        x = rot[r]
        while stk and rot[stk[-1]] < x:
            idx = stk.pop()
            ans[(start + idx) % n] = x
            print(f"  r={r} x={x}: rot[{idx}]={rot[idx]} < {x} -> ans[gốc {(start + idx) % n}] = {x}")
        stk.append(r)
    print("kết quả:", ans)
    return ans


r = trace_rotate([1, 2, 1])
print()
r = trace_rotate([5, 3, 4, 1, 2, 6, 2])

nums = [1, 2, 1]  max = 2 ở chỉ số 1
start = 2  mảng xoay rot = [1, 1, 2] (max đứng cuối)
  r=2 x=2: rot[1]=1 < 2 -> ans[gốc 0] = 2
  r=2 x=2: rot[0]=1 < 2 -> ans[gốc 2] = 2
kết quả: [2, -1, 2]

nums = [5, 3, 4, 1, 2, 6, 2]  max = 6 ở chỉ số 5
start = 6  mảng xoay rot = [2, 5, 3, 4, 1, 2, 6] (max đứng cuối)
  r=1 x=5: rot[0]=2 < 5 -> ans[gốc 6] = 5
  r=3 x=4: rot[2]=3 < 4 -> ans[gốc 1] = 4
  r=5 x=2: rot[4]=1 < 2 -> ans[gốc 3] = 2
  r=6 x=6: rot[5]=2 < 6 -> ans[gốc 4] = 6
  r=6 x=6: rot[3]=4 < 6 -> ans[gốc 2] = 6
  r=6 x=6: rot[1]=5 < 6 -> ans[gốc 0] = 6
kết quả: [6, 4, 6, 2, 6, -1, 5]


## 9. Code nộp LeetCode

Ta nộp Cách 2 (ngăn xếp đơn điệu duyệt xuôi, lưu chỉ số, 2 vòng): trực quan nhất ("đang chờ → được giải phóng").

In [19]:
class Solution:
    def nextGreaterElements(self, nums: List[int]) -> List[int]:
        n = len(nums)
        ans = [-1] * n
        stk: List[int] = []                        # chỉ số các phần tử đang chờ số lớn hơn
        for j in range(2 * n):                     # duyệt 2 vòng để xử lý tính tuần hoàn
            x = nums[j % n]
            while stk and nums[stk[-1]] < x:       # x lớn hơn thật sự -> giải phóng
                ans[stk.pop()] = x
            if j < n:                              # chỉ vòng đầu mới cho vào chờ
                stk.append(j)
        return ans


for nums, expected in examples:
    got = Solution().nextGreaterElements(nums)
    print(nums, "->", got)
    assert got == expected

[1, 2, 1] -> [2, -1, 2]
[1, 2, 3, 4, 3] -> [2, 3, 4, -1, 4]


## 10. Vì sao đúng

**Bất biến.** Trước khi xử lý vị trí `j`, ngăn xếp `stk` chứa đúng các chỉ số `i < min(j, n)` **chưa tìm được** số lớn hơn trong các giá trị `arr[i+1..j-1]` (với `arr` là mảng đôi), và `nums[stk]` **không tăng** từ đáy lên đỉnh.

**Duy trì.** Với `x = arr[j]`: vì `nums[stk]` không tăng từ đáy lên đỉnh, các chỉ số có giá trị `< x` nằm liên tiếp ở **đỉnh**; `while` pop đúng các chỉ số đó và dừng ngay khi gặp giá trị `>= x`. Mỗi chỉ số bị pop nhận `x` làm đáp án, và `x` là phần tử **đầu tiên** lớn hơn nó: nếu trước đó đã có phần tử lớn hơn, chỉ số này đã bị pop từ lúc đó rồi. Sau khi pop, mọi phần tử còn lại trong ngăn xếp đều `>= x`; push `j` (nếu `j < n`) vẫn giữ thứ tự không tăng.

**Hoàn tất.** Mỗi `i` được xét với `arr[i+1 .. i+n-1]` (tức `n - 1` phần tử sau nó theo vòng) vì `j` chạy tới `2n - 1 >= i + n - 1`. Phần tử `arr[i + n] = nums[i]` không lớn hơn chính nó nên không làm sai kết quả (và nếu `i` còn trong ngăn xếp thì chưa có số lớn hơn nào trước đó). Phần tử còn lại trong ngăn xếp không có số lớn hơn nào trong cả vòng ⇒ giữ `-1`.

**Cách 3.** Ngăn xếp giảm chặt của các giá trị bên phải: đỉnh sau khi bỏ các giá trị `<= x` là giá trị **gần nhất** bên phải **lớn hơn** `x` (các giá trị bị bỏ nhỏ hơn hoặc bằng `x` không thể là ứng viên cho `x` hay phần tử bên trái `x` mà `x` không đã phủ tốt hơn).

**Cách 4.** Xem mục 7: phần tử max ở cuối bảo đảm mọi phần tử `< max` có số lớn hơn bên phải; phần tử `= max` chắc chắn `-1`.

**Cách 5.** Đoạn bị nhảy qua toàn các phần tử `<= arr[j] <= arr[i]`, nên bỏ qua an toàn; dừng ở `j` đầu tiên có `arr[j] > arr[i]`.

## 11. Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Vét cạn | `O(n^2)` | `O(1)` |
| 2. Ngăn xếp xuôi (chỉ số, 2 vòng) | `O(n)` | `O(n)` |
| 3. Ngăn xếp ngược (giá trị, 2 vòng) | `O(n)` | `O(n)` |
| 4. Xoay theo max, 1 vòng | `O(n)` | `O(n)` |
| 5. Nhảy theo `next` | `O(n)` amortized | `O(n)` |

Không thể nhanh hơn `O(n)` vì phải sinh `n` đáp án.

## 12. Bộ test

Oracle độc lập: `oracle(nums)` xoay mảng cho từng `i` (`rest = nums[i+1:] + nums[:i]`) rồi tìm phần tử lớn hơn đầu tiên bằng `next(...)` — không dùng chỉ số `%`, không dùng ngăn xếp.

In [20]:
def oracle(nums):
    ans = []
    for i, v in enumerate(nums):
        rest = nums[i + 1:] + nums[:i]                       # đúng n - 1 phần tử theo thứ tự vòng
        ans.append(next((x for x in rest if x > v), -1))
    return ans


SOLUTIONS = [Solution(), SolutionBrute(), Solution2(), Solution3(), Solution4(), Solution5()]
print(oracle([1, 2, 1]), oracle([1, 2, 3, 4, 3]))

[2, -1, 2] [2, 3, 4, -1, 4]


In [21]:
# 12.1 Ví dụ đề + edge case
cases = [
    ([1, 2, 1], [2, -1, 2]),
    ([1, 2, 3, 4, 3], [2, 3, 4, -1, 4]),
    ([5], [-1]),                                  # 1 phần tử
    ([1, 1], [-1, -1]),
    ([1, 2], [2, -1]),
    ([2, 1], [-1, 2]),                            # phải vòng lại
    ([1, 1, 1, 1], [-1, -1, -1, -1]),             # toàn bằng nhau
    ([1, 2, 3, 4, 5], [2, 3, 4, 5, -1]),          # tăng
    ([5, 4, 3, 2, 1], [-1, 5, 5, 5, 5]),          # giảm: mọi phần tử vòng về max
    ([3, 8, 4, 1, 2], [8, -1, 8, 2, 3]),
    ([-10 ** 9, 10 ** 9, -10 ** 9], [10 ** 9, -1, 10 ** 9]),    # biên giá trị
    ([2, 2, 3, 2, 2], [3, 3, -1, 3, 3]),
    ([5, 5, 1, 5], [-1, -1, 5, -1]),              # nhiều max
]
for nums, expected in cases:
    assert oracle(nums) == expected, (nums, oracle(nums), expected)
    for sol in SOLUTIONS:
        got = sol.nextGreaterElements(list(nums))
        assert got == expected, (type(sol).__name__, nums, got, expected)
print("OK:", len(cases), "test x", len(SOLUTIONS), "lớp")

OK: 13 test x 6 lớp


In [22]:
# 12.2 Vét cạn mọi mảng độ dài <= 7 trên bảng chữ cái {0,1,2,3}
from itertools import product

count = 0
for n in range(1, 8):
    for arr in product(range(4), repeat=n):
        nums = list(arr)
        exp = oracle(nums)
        for sol in SOLUTIONS:
            assert sol.nextGreaterElements(nums) == exp, (type(sol).__name__, nums)
        count += 1
print("OK: đối chiếu", count, "mảng với oracle")

OK: đối chiếu 21844 mảng với oracle


In [23]:
# 12.3 Ngẫu nhiên có seed, đúng ràng buộc giá trị [-10^9, 10^9]; n <= 60 đối chiếu oracle
rng = random.Random(503)
for _ in range(1500):
    n = rng.randint(1, 60)
    mode = rng.choice(["wide", "narrow", "inc", "dec", "valley"])
    if mode == "wide":
        nums = [rng.randint(-10 ** 9, 10 ** 9) for _ in range(n)]
    elif mode == "narrow":
        nums = [rng.randint(-3, 3) for _ in range(n)]
    elif mode == "inc":
        nums = sorted(rng.randint(-50, 50) for _ in range(n))
    elif mode == "dec":
        nums = sorted((rng.randint(-50, 50) for _ in range(n)), reverse=True)
    else:
        half = [rng.randint(0, 20) for _ in range(n // 2 + 1)]
        nums = (sorted(half, reverse=True) + sorted(half))[:n]
    exp = oracle(nums)
    for sol in SOLUTIONS:
        assert sol.nextGreaterElements(nums) == exp, (type(sol).__name__, nums)
print("OK: 1500 mảng ngẫu nhiên n <= 60")

OK: 1500 mảng ngẫu nhiên n <= 60


In [24]:
# 12.4 n = 10^4: đối chiếu giữa các cách O(n); Cách 1 chỉ chạy trên mảng nhỏ hơn
fast = [Solution(), Solution2(), Solution3(), Solution4(), Solution5()]
rng = random.Random(5031)
big = {
    "ngẫu nhiên rộng": [rng.randint(-10 ** 9, 10 ** 9) for _ in range(10 ** 4)],
    "ngẫu nhiên hẹp": [rng.randint(0, 5) for _ in range(10 ** 4)],
    "tăng": list(range(10 ** 4)),
    "giảm": list(range(10 ** 4, 0, -1)),
    "toàn bằng nhau": [7] * 10 ** 4,
    "răng cưa": [i % 2 * 100 + (i % 7) for i in range(10 ** 4)],
}
for name, nums in big.items():
    t0 = time.perf_counter()
    res = [s.nextGreaterElements(nums) for s in fast]
    dt = time.perf_counter() - t0
    assert all(r == res[0] for r in res), name
    print(f"{name:<16}: OK ({dt * 1000:.1f} ms cho 5 cách)")

# Cách 1 và oracle trên mảng 600 phần tử: so với Cách 2
nums = [rng.randint(-10 ** 9, 10 ** 9) for _ in range(600)]
assert SolutionBrute().nextGreaterElements(nums) == oracle(nums) == Solution().nextGreaterElements(nums)
print("OK: n = 600 đối chiếu oracle và Cách 1")

ngẫu nhiên rộng : OK (17.0 ms cho 5 cách)
ngẫu nhiên hẹp  : OK (14.7 ms cho 5 cách)


tăng            : OK (12.2 ms cho 5 cách)
giảm            : OK (13.6 ms cho 5 cách)
toàn bằng nhau  : OK (9.5 ms cho 5 cách)
răng cưa        : OK (15.4 ms cho 5 cách)
OK: n = 600 đối chiếu oracle và Cách 1


## 13. Các lỗi người mới hay gặp

### Lỗi 1: coi số bằng nhau là "lớn hơn" (`<=` thay `<`)

In [25]:
def wrong_equal(nums):
    n = len(nums)
    ans = [-1] * n
    stk = []
    for j in range(2 * n):
        x = nums[j % n]
        while stk and nums[stk[-1]] <= x:          # SAI: <= thay vì <
            ans[stk.pop()] = x
        if j < n:
            stk.append(j)
    return ans


for nums in ([1, 1, 1], [2, 2, 3, 2]):
    print(nums, "sai:", wrong_equal(nums), " đúng:", Solution().nextGreaterElements(nums))

[1, 1, 1] sai: [1, 1, 1]  đúng: [-1, -1, -1]
[2, 2, 3, 2] sai: [2, 3, 3, 2]  đúng: [3, 3, -1, 3]


**Vì sao sai?** Đề cần số **lớn hơn thật sự**. Với `<=`, số bằng nhau cũng "giải phóng" nhau ⇒ `[1,1,1]` ra `[1,1,1]` thay vì `[-1,-1,-1]`. Ở bản duyệt xuôi dùng `nums[stk[-1]] < x`; ở bản duyệt ngược (Cách 3) dùng `stk[-1] <= x` để **pop** (bỏ cả giá trị bằng).

### Lỗi 2: chỉ duyệt một vòng

In [26]:
def wrong_one_pass(nums):
    n = len(nums)
    ans = [-1] * n
    stk = []
    for j in range(n):                            # SAI: chỉ n bước, không vòng lại
        while stk and nums[stk[-1]] < nums[j]:
            ans[stk.pop()] = nums[j]
        stk.append(j)
    return ans


for nums in ([1, 2, 1], [5, 4, 3, 2, 1]):
    print(nums, "sai:", wrong_one_pass(nums), " đúng:", Solution().nextGreaterElements(nums))

[1, 2, 1] sai: [2, -1, -1]  đúng: [2, -1, 2]
[5, 4, 3, 2, 1] sai: [-1, -1, -1, -1, -1]  đúng: [-1, 5, 5, 5, 5]


**Vì sao sai?** Phần tử cuối `1` của `[1,2,1]` cần "vòng lại" để thấy `2`, nhưng vòng duyệt dừng ở `j = n - 1`. Với mảng vòng phải duyệt `2n` vị trí (`j % n`) hoặc dùng các kỹ thuật tương đương (xoay theo max).

### Lỗi 3: quên `% n` khi duyệt `2n` vị trí

In [27]:
def wrong_no_mod(nums):
    n = len(nums)
    ans = [-1] * n
    stk = []
    for j in range(2 * n):
        x = nums[j]                               # SAI: thiếu % n
        while stk and nums[stk[-1]] < x:
            ans[stk.pop()] = x
        if j < n:
            stk.append(j)
    return ans


try:
    wrong_no_mod([1, 2, 1])
except IndexError as e:
    print("sai : IndexError ->", e)
print("đúng:", Solution().nextGreaterElements([1, 2, 1]))

sai : IndexError -> list index out of range
đúng: [2, -1, 2]


**Vì sao sai?** `nums` chỉ có chỉ số `0..n-1`; `nums[j]` với `j >= n` vượt biên. Phải dùng `nums[j % n]`.

### Lỗi 4: ở bản duyệt ngược, pop thiếu giá trị bằng (dùng `<` thay `<=`)

In [28]:
def wrong_backward(nums):
    n = len(nums)
    ans = [-1] * n
    stk = []
    for j in range(2 * n - 1, -1, -1):
        x = nums[j % n]
        while stk and stk[-1] < x:                # SAI: phải là <=
            stk.pop()
        if j < n and stk:
            ans[j] = stk[-1]
        stk.append(x)
    return ans


for nums in ([1, 1, 1], [2, 2, 3]):
    print(nums, "sai:", wrong_backward(nums), " đúng:", Solution().nextGreaterElements(nums))

[1, 1, 1] sai: [1, 1, 1]  đúng: [-1, -1, -1]
[2, 2, 3] sai: [2, 3, 3]  đúng: [3, 3, -1]


**Vì sao sai?** Trong bản ngược, ngăn xếp phải chứa các giá trị **lớn hơn thật sự** x. Nếu giữ giá trị bằng `x` thì đáp án cho `x` lại là chính số bằng nó (ví dụ `[1,1,1]` ra `1`). Phải pop mọi giá trị `<= x`.

### Lỗi 5: nhầm "số lớn hơn kế tiếp" với "phần tử kế tiếp"

Một số người so sánh chỉ `nums[i]` với `nums[(i + 1) % n]`. Đề yêu cầu số lớn hơn **đầu tiên** tìm được, có thể cách nhiều bước.

In [29]:
def wrong_adjacent(nums):
    n = len(nums)
    return [nums[(i + 1) % n] if nums[(i + 1) % n] > nums[i] else -1 for i in range(n)]    # SAI


nums = [1, 2, 3, 4, 3]
print("sai :", wrong_adjacent(nums))
print("đúng:", Solution().nextGreaterElements(nums))
nums = [5, 1, 2, 3, 6]
print("sai :", wrong_adjacent(nums), " đúng:", Solution().nextGreaterElements(nums))

sai : [2, 3, 4, -1, -1]
đúng: [2, 3, 4, -1, 4]
sai : [-1, 2, 3, 6, -1]  đúng: [6, 2, 3, 6, -1]


**Vì sao sai?** Với `[5,1,2,3,6]`, số lớn hơn của `5` là `6` (cách 4 bước), không phải phần tử liền sau (`1`). Phải duyệt tiếp cho tới khi gặp số lớn hơn.

## 14. Mẹo & nhận dạng pattern

- **Nhận dạng:** "phần tử lớn hơn / nhỏ hơn **gần nhất** bên phải (hoặc trái)", "ngày chờ nóng hơn", "histogram", "stock span" ⇒ **ngăn xếp đơn điệu**.
- **Hai chiều của câu hỏi:** duyệt xuôi và pop khi gặp số lớn hơn (gán đáp án cho phần tử bị pop), hoặc duyệt ngược và đọc đỉnh ngăn xếp (gán đáp án cho phần tử đang xét).
- **Mảng vòng:** duyệt **2n** vị trí với `% n`; hoặc xoay theo phần tử cực trị; hoặc nối đôi mảng.
- **Lưu chỉ số hay giá trị?** Khi cần ghi kết quả vào vị trí của phần tử bị pop ⇒ lưu **chỉ số**. Khi chỉ cần giá trị ứng viên ⇒ lưu **giá trị**.
- **Strict hay không?** "lớn hơn" ⇒ `<` khi pop ở bản xuôi (hoặc `<=` ở bản ngược). Luôn kiểm tra bằng test có số trùng.
- **Chỉ vòng đầu push.** Vòng hai chỉ để "giải phóng" — push thêm chỉ lãng phí.
- Mỗi phần tử vào/ra ngăn xếp tối đa một lần ⇒ tổng `O(n)` dù có `while` lồng trong `for`.

## 15. Biến thể

1. **Next Greater Element I (496):** hai mảng, tìm trong mảng thứ hai; không vòng.
2. **Daily Temperatures (739):** thay vì giá trị, trả **khoảng cách** tới ngày nóng hơn (bài tập 5 làm bản vòng).
3. **Next Smaller Element (vòng):** đảo dấu so sánh.
4. **Online Stock Span (901):** ngăn xếp đơn điệu lưu cặp (giá, span).
5. **Largest Rectangle in Histogram (84):** ngăn xếp đơn điệu tìm biên trái/phải.
6. **Trapping Rain Water (42):** ngăn xếp đơn điệu hoặc hai con trỏ.

In [30]:
# Biến thể 3: số nhỏ hơn kế tiếp trong mảng vòng (đảo dấu so sánh), kèm oracle
def next_smaller_circular(nums):
    n = len(nums)
    ans = [-1] * n
    stk = []
    for j in range(2 * n):
        x = nums[j % n]
        while stk and nums[stk[-1]] > x:
            ans[stk.pop()] = x
        if j < n:
            stk.append(j)
    return ans


def next_smaller_oracle(nums):
    return [next((x for x in nums[i + 1:] + nums[:i] if x < v), -1) for i, v in enumerate(nums)]


print(next_smaller_circular([3, 8, 4, 1, 2]), next_smaller_oracle([3, 8, 4, 1, 2]))
rng = random.Random(1)
for _ in range(300):
    a = [rng.randint(-4, 4) for _ in range(rng.randint(1, 12))]
    assert next_smaller_circular(a) == next_smaller_oracle(a)
print("OK")

[1, 4, 1, -1, 1] [1, 4, 1, -1, 1]
OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | với mỗi `i` quét tối đa `n-1` bước vòng | `O(n^2)` | `O(1)` | oracle, `n` nhỏ |
| 2. Ngăn xếp xuôi (chỉ số) | "đang chờ → giải phóng", 2 vòng | `O(n)` | `O(n)` | **nộp bài**, trực quan |
| 3. Ngăn xếp ngược (giá trị) | ngăn xếp ứng viên, 2 vòng | `O(n)` | `O(n)` | quen với "đọc đỉnh ngăn xếp" |
| 4. Xoay theo max | max ở cuối ⇒ không cần vòng | `O(n)` | `O(n)` | muốn tránh xử lý vòng |
| 5. Nhảy theo `next` | pointer jumping trên mảng đôi | `O(n)` amortized | `O(n)` | hiểu "ngăn xếp ẩn trong con trỏ" |

In [31]:
rng = random.Random(8)
nums = [rng.randint(-10 ** 9, 10 ** 9) for _ in range(10 ** 4)]
print("n = 10^4 ngẫu nhiên:")
for name, sol in [("2 xuôi", Solution2()), ("3 ngược", Solution3()), ("4 xoay", Solution4()), ("5 nhảy", Solution5())]:
    t0 = time.perf_counter()
    for _ in range(20):
        sol.nextGreaterElements(nums)
    print(f"{name:<8}: {(time.perf_counter() - t0) / 20 * 1000:.2f} ms")
small = nums[:1500]
t0 = time.perf_counter()
SolutionBrute().nextGreaterElements(small)
print(f"vét cạn trên 1500 phần tử: {(time.perf_counter() - t0) * 1000:.1f} ms (n^2 tăng rất nhanh)")

n = 10^4 ngẫu nhiên:


2 xuôi  : 2.62 ms


3 ngược : 3.20 ms


4 xoay  : 2.99 ms


5 nhảy  : 3.34 ms
vét cạn trên 1500 phần tử: 0.9 ms (n^2 tăng rất nhanh)


## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `Solution().nextGreaterElements([4, 4, 4])`; (b) `Solution().nextGreaterElements([3, 1])`; (c) `(3 + 2) % 4`; (d) sau khi chạy trace với `[2, 1, 2]`, ngăn xếp cuối cùng chứa chỉ số nào (Cách 2)?

**Bài 2 (tính tay).** Với `nums = [3, 8, 4, 1, 2]`, lập bảng như mục 4 (dãy đi tiếp theo vòng và số lớn hơn đầu tiên).

**Bài 3 (điền TODO).** Hoàn thành `ng_todo(nums)` theo Cách 2.

In [32]:
def ng_todo(nums):
    # TODO 1: n = len(nums); ans = [-1] * n; stk = []
    # TODO 2: for j in range(2 * n): x = nums[j % n]
    # TODO 3: while stk and nums[stk[-1]] < x: ans[stk.pop()] = x
    # TODO 4: nếu j < n thì stk.append(j)
    # TODO 5: return ans
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `ng_backward(nums)` theo Cách 3 (ngăn xếp giá trị, duyệt ngược `2n-1 → 0`) **không nhìn lại** mục 7. Kiểm tra với `[1,2,1]` và `[1,2,3,4,3]`.

**Bài 5 (biến thể / mini challenge).** Viết `next_greater_distance(nums)`: với mỗi phần tử, trả về **số bước** cần đi (theo vòng) tới số lớn hơn đầu tiên; nếu không có thì `0`. Ví dụ `[1,2,1]` → `[1,0,2]`. Kiểm chứng với oracle vét cạn.

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ).** Bài 1: số bằng nhau **không** tính là lớn hơn; `(3+2) % 4 = 1`. Bài 3: nhớ chỉ `append` khi `j < n`.

**Mức 2 (vừa).** Bài 2: với `3`: dãy đi tiếp `8,4,1,2` ⇒ `8`; với `1`: `2,3,8,4` ⇒ `2`; với `2`: `3,8,4,1` ⇒ `3`; với `8`: không có số lớn hơn ⇒ `-1`; với `4`: `1,2,3,8` ⇒ `8`. Bài 4: `while stk and stk[-1] <= x: stk.pop()`; chỉ ghi `ans[j]` khi `j < n`.

**Mức 3 (sát đáp án).** Bài 5: ở Cách 2, khi pop chỉ số `i` tại vị trí `j`, khoảng cách là `j - i` (với `j` chạy trên mảng đôi, `j - i <= n - 1`). Khởi tạo `ans = [0] * n`.

## 19. Đáp án tham khảo cho bài tập

In [33]:
# Bài 1
assert Solution().nextGreaterElements([4, 4, 4]) == [-1, -1, -1]
assert Solution().nextGreaterElements([3, 1]) == [-1, 3]
assert (3 + 2) % 4 == 1
# [2,1,2]: Cách 2 -> kết thúc với ngăn xếp [0, 2] (hai số 2, không ai lớn hơn)
n_ = 3
stk, nums_ = [], [2, 1, 2]
for j in range(2 * n_):
    x = nums_[j % n_]
    while stk and nums_[stk[-1]] < x:
        stk.pop()
    if j < n_:
        stk.append(j)
assert stk == [0, 2], stk

# Bài 2
assert oracle([3, 8, 4, 1, 2]) == [8, -1, 8, 2, 3]
print("bài 1-2 OK")

bài 1-2 OK


In [34]:
# Bài 3
def ng_todo_answer(nums):
    n = len(nums)
    ans = [-1] * n
    stk = []
    for j in range(2 * n):
        x = nums[j % n]
        while stk and nums[stk[-1]] < x:
            ans[stk.pop()] = x
        if j < n:
            stk.append(j)
    return ans


for nums, expected in examples:
    assert ng_todo_answer(nums) == expected
print("bài 3 OK")

bài 3 OK


In [35]:
# Bài 4
def ng_backward(nums):
    n = len(nums)
    ans = [-1] * n
    stk = []
    for j in range(2 * n - 1, -1, -1):
        x = nums[j % n]
        while stk and stk[-1] <= x:
            stk.pop()
        if j < n and stk:
            ans[j] = stk[-1]
        stk.append(x)
    return ans


for nums, expected in examples:
    assert ng_backward(nums) == expected
rng = random.Random(5032)
for _ in range(300):
    a = [rng.randint(-3, 3) for _ in range(rng.randint(1, 15))]
    assert ng_backward(a) == oracle(a)
print("bài 4 OK")

bài 4 OK


In [36]:
# Bài 5
def next_greater_distance(nums):
    n = len(nums)
    ans = [0] * n
    stk = []
    for j in range(2 * n):
        x = nums[j % n]
        while stk and nums[stk[-1]] < x:
            i = stk.pop()
            ans[i] = j - i
        if j < n:
            stk.append(j)
    return ans


def distance_oracle(nums):
    n = len(nums)
    res = []
    for i in range(n):
        d = 0
        for step in range(1, n):
            if nums[(i + step) % n] > nums[i]:
                d = step
                break
        res.append(d)
    return res


assert next_greater_distance([1, 2, 1]) == [1, 0, 2]
rng = random.Random(5033)
for _ in range(500):
    a = [rng.randint(-3, 3) for _ in range(rng.randint(1, 15))]
    assert next_greater_distance(a) == distance_oracle(a), a
print("bài 5 OK")

bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (ngăn xếp đơn điệu, mảng vòng)

```python
class Solution:
    def nextGreaterElements(self, nums: List[int]) -> List[int]:
        n = len(nums)
        ans = [-1] * n
        stk = []                              # chỉ số đang chờ, nums[stk] không tăng
        for j in range(2 * n):
            x = nums[j % n]
            while stk and nums[stk[-1]] < x:
                ans[stk.pop()] = x
            if j < n:
                stk.append(j)
        return ans
```

**Checklist**

- [ ] "Lớn hơn" là `>` thật sự: pop khi `nums[stk[-1]] < x` (bản xuôi) / pop khi `stk[-1] <= x` (bản ngược)?
- [ ] Duyệt đủ `2n` vị trí và dùng `% n`?
- [ ] Chỉ `append` khi `j < n`?
- [ ] Ngăn xếp lưu chỉ số (để ghi `ans`), `ans` khởi tạo `-1`?
- [ ] Đã thử `[5]`, `[1,1,1]`, `[2,1]`, tăng, giảm, nhiều số lớn nhất bằng nhau, `n = 10^4`, giá trị `±10^9`?
- [ ] Đã đối chiếu với oracle (xoay mảng + `next`) trên mảng nhỏ?